# PPEF Enrollment — Delaware peek

Scope: Medicare FFS Public Provider Enrollment. Clean NPI -> PAC ID spine, carries state + provider type.
Source: `data/sources/provider_registry/reassignment/Medicare_FFS_Public_Provider_Enrollment_Q3_2026.zip` (~337 MB).
Reading: chunked pandas, filtered to `STATE_CD` = DE.

In [3]:
import zipfile
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
ZIP = ROOT / "data/sources/provider_registry/reassignment/Medicare_FFS_Public_Provider_Enrollment_Q3_2026.zip"
MEMBER = "Medicare_FFS_Public_Provider_Enrollment_Q3_2026.csv"
STATE_COL, STATE = "STATE_CD", "DE"
ENC = "latin-1"  # this file is not UTF-8 (contains 0xA0); latin-1 reads every byte safely
with zipfile.ZipFile(ZIP) as z, z.open(MEMBER) as f:
    sample = pd.read_csv(f, nrows=5, dtype=str, encoding=ENC)
print(sample.shape[1], "columns")
sample.head()

11 columns


,NPI,MULTIPLE_NPI_FLAG,PECOS_ASCT_CNTL_ID,ENRLMT_ID,PROVIDER_TYPE_CD,PROVIDER_TYPE_DESC,STATE_CD,FIRST_NAME,MDL_NAME,LAST_NAME,ORG_NAME
0,1003879883,N,8022920719,I20031103000001,14-16,PRACTITIONER - OBSTETRICS/GYNECOLOGY,PR,ANTONIO,NaN,ALVAREZ RODRIGUEZ,NaN
1,1003976986,N,7113839812,I20031103000005,14-68,PRACTITIONER - CLINICAL PSYCHOLOGIST,PA,CHRISTOPHER,J,ZIEGLER,NaN
2,1407802119,N,8022920727,I20031103000007,14-93,PRACTITIONER - EMERGENCY MEDICINE,PA,KADISHA,B,RAPP,NaN
3,1831165075,N,5193637890,I20031103000013,14-16,PRACTITIONER - OBSTETRICS/GYNECOLOGY,PR,JORGE,A,OSTOLAZA BEY,NaN
4,1851357214,N,2466364161,I20031103000014,14-30,PRACTITIONER - DIAGNOSTIC RADIOLOGY,KY,RHONDA,G,GRISSOM,NaN


In [4]:
# Scope to Delaware. Chunked scan (~337 MB). latin-1 encoding (see setup cell).
parts = []
with zipfile.ZipFile(ZIP) as z, z.open(MEMBER) as f:
    for chunk in pd.read_csv(f, dtype=str, chunksize=200_000, encoding=ENC):
        parts.append(chunk[chunk[STATE_COL] == STATE])
de = pd.concat(parts, ignore_index=True)
de.shape

(12778, 11)

In [5]:
# Enrollment spine: NPI, PAC ID, provider type mix.
print("rows:", len(de), "| unique NPIs:", de["NPI"].nunique(), "| unique PAC IDs:", de["PECOS_ASCT_CNTL_ID"].nunique())
de["PROVIDER_TYPE_DESC"].value_counts().head(20)

rows: 12778 | unique NPIs: 12679 | unique PAC IDs: 12435


PROVIDER_TYPE_DESC
PRACTITIONER - NURSE PRACTITIONER                               1919
PART B SUPPLIER - CLINIC/GROUP PRACTICE                         1227
PRACTITIONER - PHYSICIAN ASSISTANT                               942
PRACTITIONER - PHYSICAL THERAPIST IN PRIVATE PRACTICE            730
PRACTITIONER - INTERNAL MEDICINE                                 671
PRACTITIONER - FAMILY PRACTICE                                   445
PRACTITIONER - CLINICAL SOCIAL WORKER                            428
PRACTITIONER - CERTIFIED REGISTERED NURSE ANESTHETIST (CRNA)     403
PRACTITIONER - EMERGENCY MEDICINE                                363
PRACTITIONER - DIAGNOSTIC RADIOLOGY                              320
PRACTITIONER - MENTAL HEALTH COUNSELOR                           312
PRACTITIONER - ANESTHESIOLOGY                                    223
PRACTITIONER - CHIROPRACTIC                                      197
PRACTITIONER - OBSTETRICS/GYNECOLOGY                             195
PRACTITIONER - 

## Notes
- One row per enrollment; `PECOS_ASCT_CNTL_ID` = PAC ID. One PAC ID maps to many enrollments.
- Carries `STATE_CD` + `PROVIDER_TYPE_DESC`, so DE + provider-type filtering is native here (HISTORY 005).
- `PROVIDER_TYPE_DESC` is prefixed `PRACTITIONER - ...`; org enrollments show `PART B SUPPLIER - CLINIC/GROUP PRACTICE`.
- **Quirk:** file is `latin-1`, not UTF-8 (byte 0xA0). Read with `encoding="latin-1"`.
- Pairs with the Revalidation xref (the individual->group link) for Q4.